# SER via CNN + BiLSTM (from scratch)

**Pipeline:**
1. Use existing RAVDESS + CREMA-D download (or re-download if fresh runtime)
2. Extract log-mel spectrograms (not MFCC — richer, keeps time dimension)
3. Train CNN + BiLSTM from scratch with heavy augmentation
4. Evaluate, save, live mic test

**Architecture:** 2D CNN over time-freq -> reshape to sequence -> BiLSTM -> Dense -> Softmax(6)

**Target:** ≥ 60% speaker-independent test accuracy. If we beat YAMNet (54.5%), we switch to this.

## Step 0 — Mount Drive and check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/BitirmeProjesi/ser_cnn_bilstm'
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')

import tensorflow as tf
print(f'TF: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPU: {gpus}')

Mounted at /content/drive
Output dir: /content/drive/MyDrive/BitirmeProjesi/ser_cnn_bilstm
TF: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Step 1 — Install deps and download datasets

Skip this cell if RAVDESS and CREMA-D are already in /content from the YAMNet notebook.

In [ ]:
!pip install -q librosa soundfile tqdm audiomentations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 18.5 MB/s eta 0:00:00


In [ ]:
import os
os.environ['KAGGLE_USERNAME'] = 'ersnkara'
os.environ['KAGGLE_KEY'] = 'KGAT_2f11f9cf7a6a07d02e5bf6bade1f1bf1'

!pip install -q kaggle
!kaggle datasets download -d ejlok1/cremad -p /content --unzip
!mkdir -p /content/CREMA-D/AudioWAV
!mv /content/AudioWAV/* /content/CREMA-D/AudioWAV/ 2>/dev/null || true
!ls /content/CREMA-D/AudioWAV | wc -l

Dataset URL: https://www.kaggle.com/datasets/ejlok1/cremad
License(s): ODC Attribution License (ODC-By)
100% 451M/451M [00:05<00:00, 91.8MB/s]

7442


In [ ]:
# RAVDESS
if not os.path.exists('/content/RAVDESS'):
    !wget -q -O /content/RAVDESS.zip https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip
    !mkdir -p /content/RAVDESS && unzip -q /content/RAVDESS.zip -d /content/RAVDESS
    print('RAVDESS downloaded')
else:
    print('RAVDESS already present')

# CREMA-D via Kaggle
if not os.path.exists('/content/CREMA-D/AudioWAV'):
    print('CREMA-D not found. Run the Kaggle download cells from the YAMNet notebook first.')
    print('Required: /content/CREMA-D/AudioWAV/ with 7442 .wav files')
else:
    n_crema = len(os.listdir('/content/CREMA-D/AudioWAV'))
    print(f'CREMA-D present: {n_crema} files')

RAVDESS downloaded
CREMA-D present: 7442 files


## Step 2 — Build unified clip index (same logic as YAMNet notebook)

In [ ]:
import os
import pandas as pd
import numpy as np

CLASSES = ['Angry', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

RAVDESS_EMOTION_MAP = {
    '01': 'Neutral', '03': 'Happy', '04': 'Sad',
    '05': 'Angry', '06': 'Fear', '08': 'Surprise',
}
CREMA_EMOTION_MAP = {
    'NEU': 'Neutral', 'HAP': 'Happy', 'SAD': 'Sad',
    'ANG': 'Angry', 'FEA': 'Fear',
}

rows = []
ravdess_root = '/content/RAVDESS'
for actor_dir in sorted(os.listdir(ravdess_root)):
    actor_path = os.path.join(ravdess_root, actor_dir)
    if not os.path.isdir(actor_path):
        continue
    for fn in sorted(os.listdir(actor_path)):
        if not fn.endswith('.wav'):
            continue
        parts = fn.replace('.wav', '').split('-')
        emo_code = parts[2]; actor = parts[6]
        if emo_code not in RAVDESS_EMOTION_MAP:
            continue
        rows.append({
            'path': os.path.join(actor_path, fn),
            'emotion': RAVDESS_EMOTION_MAP[emo_code],
            'speaker': f'RAV_{actor}', 'source': 'RAVDESS',
        })

crema_root = '/content/CREMA-D/AudioWAV'
for fn in sorted(os.listdir(crema_root)):
    if not fn.endswith('.wav'):
        continue
    parts = fn.replace('.wav', '').split('_')
    if len(parts) < 3:
        continue
    actor = parts[0]; emo_code = parts[2]
    if emo_code not in CREMA_EMOTION_MAP:
        continue
    rows.append({
        'path': os.path.join(crema_root, fn),
        'emotion': CREMA_EMOTION_MAP[emo_code],
        'speaker': f'CRE_{actor}', 'source': 'CREMA-D',
    })

df = pd.DataFrame(rows)
df['label'] = df['emotion'].map(CLASS_TO_IDX)
print(f'Total clips: {len(df)} | Speakers: {df["speaker"].nunique()}')
print(df['emotion'].value_counts())

Total clips: 7227 | Speakers: 115
emotion
Happy       1463
Sad         1463
Fear        1463
Angry       1463
Neutral     1183
Surprise     192
Name: count, dtype: int64


## Step 3 — Speaker-independent split (same seed as YAMNet notebook for fair comparison)

In [ ]:
SEED = 42
rng = np.random.default_rng(SEED)
speakers = sorted(df['speaker'].unique())
rng.shuffle(speakers)
n = len(speakers); n_train = int(0.8 * n); n_val = int(0.1 * n)
train_speakers = set(speakers[:n_train])
val_speakers = set(speakers[n_train:n_train + n_val])
test_speakers = set(speakers[n_train + n_val:])

df['split'] = df['speaker'].apply(
    lambda s: 'train' if s in train_speakers else ('val' if s in val_speakers else 'test'))

print(df.groupby(['split', 'emotion']).size().unstack(fill_value=0))
print(f'\nTrain: {(df["split"]=="train").sum()} | Val: {(df["split"]=="val").sum()} | Test: {(df["split"]=="test").sum()}')

emotion  Angry  Fear  Happy  Neutral   Sad  Surprise
split                                               
test       161   161    161      135   161         8
train     1166  1166   1166      940  1166       160
val        136   136    136      108   136        24

Train: 5764 | Val: 676 | Test: 787


## Step 4 — Mel-spectrogram parameters

- **Sample rate:** 16000 Hz (same as live pipeline)
- **Duration:** 3.0 seconds — clips longer than this are center-cropped, shorter are zero-padded. This gives a fixed input shape, critical for batch training.
- **n_mels = 128** — 128 frequency bins, standard for speech
- **n_fft = 1024, hop_length = 256** — gives ~62.5 frames/sec → ~188 frames for 3s
- **Log-scaling + normalization** per-sample for stability

In [ ]:
import librosa

SR = 16000
DURATION = 3.0
TARGET_LEN = int(SR * DURATION)
N_MELS = 128
N_FFT = 1024
HOP = 256
# Expected output shape: (N_MELS, ceil(TARGET_LEN / HOP)) = (128, 188)

def load_waveform(path, sr=SR):
    y, _ = librosa.load(path, sr=sr, mono=True)
    y, _ = librosa.effects.trim(y, top_db=30)
    return y.astype(np.float32)

def fix_length(y, target_len=TARGET_LEN):
    if len(y) > target_len:
        # Center crop
        start = (len(y) - target_len) // 2
        return y[start:start + target_len]
    elif len(y) < target_len:
        return np.pad(y, (0, target_len - len(y)))
    return y

def waveform_to_logmel(y):
    mel = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS, power=2.0)
    logmel = librosa.power_to_db(mel, ref=np.max)  # dB scale
    # Per-sample normalize to roughly [-1, 1]
    logmel = (logmel - logmel.mean()) / (logmel.std() + 1e-6)
    return logmel.astype(np.float32)

# Smoke test
sample_y = fix_length(load_waveform(df.iloc[0]['path']))
sample_mel = waveform_to_logmel(sample_y)
print(f'Waveform shape: {sample_y.shape} | Mel shape: {sample_mel.shape}')
INPUT_SHAPE = sample_mel.shape + (1,)
print(f'Model input shape: {INPUT_SHAPE}')

Waveform shape: (48000,) | Mel shape: (128, 188)
Model input shape: (128, 188, 1)


## Step 5 — Pre-load raw waveforms for all clips

We cache **waveforms** (not spectrograms) so we can apply augmentation on raw audio at training time. This is what makes augmentation meaningful — if we cached spectrograms we'd only have SpecAugment.

~8800 clips × 3s × 16000Hz × 4 bytes = ~1.7 GB. Fits in Colab RAM.

In [ ]:
from tqdm.auto import tqdm

WAVE_CACHE = os.path.join(OUTPUT_DIR, 'waveforms_cache.npz')

if os.path.exists(WAVE_CACHE):
    print('Loading cached waveforms...')
    cache = np.load(WAVE_CACHE)
    W_all = cache['W']
    y_all = cache['y']
    splits = cache['splits']
else:
    print(f'Loading and fixing length for {len(df)} clips...')
    W_list, y_list, split_list = [], [], []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        try:
            w = fix_length(load_waveform(row['path']))
            W_list.append(w)
            y_list.append(row['label'])
            split_list.append(row['split'])
        except Exception as e:
            print(f'Skip {row["path"]}: {e}')
    W_all = np.stack(W_list).astype(np.float32)
    y_all = np.array(y_list, dtype=np.int32)
    splits = np.array(split_list)
    np.savez_compressed(WAVE_CACHE, W=W_all, y=y_all, splits=splits)
    print(f'Saved waveform cache.')

train_mask = splits == 'train'
val_mask = splits == 'val'
test_mask = splits == 'test'

W_train = W_all[train_mask]; y_train = y_all[train_mask]
W_val = W_all[val_mask];     y_val = y_all[val_mask]
W_test = W_all[test_mask];   y_test = y_all[test_mask]

print(f'Train: {W_train.shape} | Val: {W_val.shape} | Test: {W_test.shape}')

Loading and fixing length for 7227 clips...


  0%|          | 0/7227 [00:00<?, ?it/s]

Saved waveform cache.
Train: (5764, 48000) | Val: (676, 48000) | Test: (787, 48000)


## Step 6 — Data augmentation pipeline

Applied **only to training data**, at batch time (not pre-cached). Each epoch sees different variants of each clip.

- **AddGaussianNoise** — simulates mic/room noise
- **TimeStretch** — slight speed change without pitch change
- **PitchShift** — simulates different vocal tract sizes
- **Shift** — shifts audio in time
- **SpecAugment** (time & freq masking) applied on the spectrogram side

In [ ]:
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift, Shift

augment_wave = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.5),
    TimeStretch(min_rate=0.9, max_rate=1.1, p=0.4),
    PitchShift(min_semitones=-2, max_semitones=2, p=0.4),
    Shift(min_shift=-0.1, max_shift=0.1, p=0.4),
])

def spec_augment(mel, time_mask_prob=0.5, freq_mask_prob=0.5,
                 time_mask_max=20, freq_mask_max=15):
    mel = mel.copy()
    F, T = mel.shape
    if np.random.rand() < freq_mask_prob:
        f = np.random.randint(1, freq_mask_max)
        f0 = np.random.randint(0, max(1, F - f))
        mel[f0:f0+f, :] = mel.mean()
    if np.random.rand() < time_mask_prob:
        t = np.random.randint(1, time_mask_max)
        t0 = np.random.randint(0, max(1, T - t))
        mel[:, t0:t0+t] = mel.mean()
    return mel

def preprocess_sample(w, label, training):
    if training:
        w = augment_wave(samples=w, sample_rate=SR)
        w = fix_length(w)  # augmentation may change length
    mel = waveform_to_logmel(w)
    if training:
        mel = spec_augment(mel)
    mel = mel[..., np.newaxis]  # add channel dim
    return mel, label

In [ ]:
# Build tf.data pipelines with the preprocess_sample as a numpy_function

BATCH_SIZE = 32

def make_dataset(W, y, training, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices((W, y))
    if training:
        ds = ds.shuffle(buffer_size=len(W), reshuffle_each_iteration=True)

    def _py_preprocess(w, lbl):
        w_np = w.numpy()
        lbl_np = lbl.numpy()
        mel, lbl_out = preprocess_sample(w_np, lbl_np, training=training)
        return mel.astype(np.float32), np.int32(lbl_out)

    def _tf_preprocess(w, lbl):
        mel, lbl = tf.py_function(_py_preprocess, [w, lbl], [tf.float32, tf.int32])
        mel.set_shape(INPUT_SHAPE)
        lbl.set_shape(())
        return mel, lbl

    ds = ds.map(_tf_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(W_train, y_train, training=True)
val_ds = make_dataset(W_val, y_val, training=False)
test_ds = make_dataset(W_test, y_test, training=False)

# Smoke test
for xb, yb in train_ds.take(1):
    print(f'Batch X: {xb.shape} | y: {yb.shape}')

Batch X: (32, 128, 188, 1) | y: (32,)


## Step 7 — CNN + BiLSTM architecture

- **CNN block:** 3 Conv2D layers on (freq × time × 1), each followed by BatchNorm + ReLU + MaxPool. Pool on freq, keep time mostly intact.
- **Reshape:** (freq/8 × time/2 × 128) → (time/2, freq/8 * 128) to feed BiLSTM as sequence.
- **BiLSTM block:** 2 layers of Bidirectional LSTM, 64 units each.
- **Head:** GlobalAvgPool over time → Dense(128) → Dense(6).

In [ ]:
from tensorflow.keras import layers, models

NUM_CLASSES = len(CLASSES)

def build_model(input_shape, num_classes):
    inp = layers.Input(shape=input_shape)  # (128, 188, 1)

    # CNN block 1
    x = layers.Conv2D(32, (3, 3), padding='same')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPool2D((2, 2))(x)  # halve both freq and time
    x = layers.Dropout(0.2)(x)

    # CNN block 2
    x = layers.Conv2D(64, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPool2D((2, 1))(x)  # halve freq only
    x = layers.Dropout(0.2)(x)

    # CNN block 3
    x = layers.Conv2D(128, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPool2D((2, 1))(x)  # halve freq only
    x = layers.Dropout(0.3)(x)

    # Reshape to sequence for LSTM: (batch, time, features)
    # After above pools: freq = 128/8 = 16, time = 188/2 = 94, channels = 128
    # Permute so time is the sequence axis
    x = layers.Permute((2, 1, 3))(x)  # (batch, time, freq, channels)
    shape = x.shape
    x = layers.Reshape((shape[1], shape[2] * shape[3]))(x)  # (batch, time, freq*channels)

    # BiLSTM block
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
    x = layers.Dropout(0.3)(x)

    # Pool over time
    x = layers.GlobalAveragePooling1D()(x)

    # Head
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inp, out)
    return model

model = build_model(INPUT_SHAPE, NUM_CLASSES)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()
print(f'\nTotal params: {model.count_params():,}')

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 188, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 188, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 188, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 128, 188, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 94, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 94, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 94, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 94, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 64, 94, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 94, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32, 94, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 94, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 94, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 32, 94, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 94, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16, 94, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ permute (Permute)               │ (None, 94, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 94, 2048)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 94, 128)        │     1,081,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 94, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 94, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 94, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │             

 Total params: 1,291,526 (4.93 MB)

 Trainable params: 1,291,078 (4.93 MB)

 Non-trainable params: 448 (1.75 KB)


Total params: 1,291,526


## Step 8 — Train

Typical run: 60-100 epochs, ~60-90s per epoch with GPU + augmentation on CPU. Expect 1-2 hours.

In [ ]:
from tensorflow.keras import callbacks
from sklearn.utils.class_weight import compute_class_weight

cw = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train)
class_weight = {i: float(w) for i, w in enumerate(cw)}
print(f'Class weights: {class_weight}')

CKPT_PATH = os.path.join(OUTPUT_DIR, 'ser_cnn_bilstm_best.keras')

cb_list = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=15,
                            restore_best_weights=True, mode='max'),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5),
    callbacks.ModelCheckpoint(CKPT_PATH, monitor='val_accuracy', save_best_only=True, mode='max'),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=120,
    class_weight=class_weight,
    callbacks=cb_list,
    verbose=2,
)

Class weights: {0: 0.8238993710691824, 1: 0.8238993710691824, 2: 0.8238993710691824, 3: 1.021985815602837, 4: 0.8238993710691824, 5: 6.004166666666666}
Epoch 1/120
181/181 - 234s - 1s/step - accuracy: 0.2658 - loss: 1.6233 - val_accuracy: 0.2678 - val_loss: 1.6282 - learning_rate: 0.0010
Epoch 2/120
181/181 - 255s - 1s/step - accuracy: 0.3369 - loss: 1.4508 - val_accuracy: 0.3935 - val_loss: 1.4607 - learning_rate: 0.0010
Epoch 3/120
181/181 - 216s - 1s/step - accuracy: 0.3683 - loss: 1.4040 - val_accuracy: 0.3802 - val_loss: 1.4602 - learning_rate: 0.0010
Epoch 4/120
181/181 - 263s - 1s/step - accuracy: 0.4002 - loss: 1.3406 - val_accuracy: 0.4083 - val_loss: 1.4496 - learning_rate: 0.0010
Epoch 5/120
181/181 - 217s - 1s/step - accuracy: 0.4152 - loss: 1.2917 - val_accuracy: 0.4024 - val_loss: 1.4335 - learning_rate: 0.0010
Epoch 6/120
181/181 - 268s - 1s/step - accuracy: 0.4327 - loss: 1.2627 - val_accuracy: 0.4660 - val_loss: 1.3067 - learning_rate: 0.0010
Epoch 7/120
181/181 - 218s

In [ ]:
# === RECOVERY: Load checkpoint, evaluate, save live-testable model ===

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import tensorflow as tf
import librosa

OUTPUT_DIR = '/content/drive/MyDrive/BitirmeProjesi/ser_cnn_bilstm'
CKPT_PATH = os.path.join(OUTPUT_DIR, 'ser_cnn_bilstm_best.keras')
WAVE_CACHE = os.path.join(OUTPUT_DIR, 'waveforms_cache.npz')

print(f'Checkpoint exists: {os.path.exists(CKPT_PATH)}')
print(f'Wave cache exists: {os.path.exists(WAVE_CACHE)}')

if os.path.exists(CKPT_PATH):
    size_mb = os.path.getsize(CKPT_PATH) / (1024 * 1024)
    print(f'Checkpoint size: {size_mb:.1f} MB')

## Step 9 — Evaluate

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(history.history['accuracy'], label='train')
a1.plot(history.history['val_accuracy'], label='val')
a1.set_title('Accuracy'); a1.legend(); a1.grid(True)
a2.plot(history.history['loss'], label='train')
a2.plot(history.history['val_loss'], label='val')
a2.set_title('Loss'); a2.legend(); a2.grid(True)
plt.show()

val_loss, val_acc = model.evaluate(val_ds, verbose=0)
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f'Val  accuracy: {val_acc:.4f}')
print(f'Test accuracy: {test_acc:.4f}')

y_pred_list, y_true_list = [], []
for xb, yb in test_ds:
    p = model.predict(xb, verbose=0).argmax(axis=1)
    y_pred_list.append(p)
    y_true_list.append(yb.numpy())
y_pred = np.concatenate(y_pred_list)
y_true = np.concatenate(y_true_list)

report = classification_report(y_true, y_pred, target_names=CLASSES, digits=3)
print(report)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title(f'CNN+BiLSTM Test CM (acc={test_acc:.3f})')
plt.show()

## Step 10 — Save artifacts

In [ ]:
import json

FINAL_PATH = os.path.join(OUTPUT_DIR, 'ser_cnn_bilstm.keras')
model.save(FINAL_PATH)
print(f'Saved: {FINAL_PATH}')

meta = {
    'classes': CLASSES,
    'target_sample_rate': SR,
    'duration_seconds': DURATION,
    'n_mels': N_MELS,
    'n_fft': N_FFT,
    'hop_length': HOP,
    'input_shape': list(INPUT_SHAPE),
    'val_accuracy': float(val_acc),
    'test_accuracy': float(test_acc),
    'sources': ['RAVDESS', 'CREMA-D'],
    'architecture': 'log-mel (128x188) -> 3xConv2D+BN -> BiLSTM x2 -> GAP -> Dense(128) -> Softmax(6)',
    'params': int(model.count_params()),
}
with open(os.path.join(OUTPUT_DIR, 'ser_cnn_bilstm_meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)
with open(os.path.join(OUTPUT_DIR, 'ser_cnn_bilstm_report.txt'), 'w') as f:
    f.write(report)

print('All artifacts saved.')

## Step 11 — Live mic test

In [ ]:
from IPython.display import Javascript, display
from google.colab import output as colab_output
import base64

RECORD = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));
const b2t = blob => new Promise(r => { const f = new FileReader(); f.onloadend = e => r(e.target.result); f.readAsDataURL(blob); });
var record = time => new Promise(async resolve => {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  const recorder = new MediaRecorder(stream);
  const chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
  await sleep(time);
  recorder.stop();
  await new Promise(r => recorder.onstop = r);
  const blob = new Blob(chunks);
  const b64 = await b2t(blob);
  resolve(b64);
});
"""

def record_audio(seconds=4):
    display(Javascript(RECORD))
    b64 = colab_output.eval_js(f'record({seconds*1000})')
    data = base64.b64decode(b64.split(',')[1])
    with open('/content/mic.webm', 'wb') as f:
        f.write(data)
    !ffmpeg -y -i /content/mic.webm -ar 16000 -ac 1 /content/mic.wav -loglevel error
    return '/content/mic.wav'

In [ ]:
print('Speak for 4 seconds...')
path = record_audio(4)
w = fix_length(load_waveform(path))
mel = waveform_to_logmel(w)[..., np.newaxis]
probs = model.predict(mel[np.newaxis, ...], verbose=0)[0]

for c, p in sorted(zip(CLASSES, probs), key=lambda x: -x[1]):
    bar = '█' * int(p * 40)
    print(f'{c:10s} {p:.3f} {bar}')

## What to check vs YAMNet

| Metric | YAMNet v1 | CNN+BiLSTM |
|--------|-----------|------------|
| Test accuracy | 0.545 | ? |
| Fear recall | 0.34 | ? |
| Happy recall | 0.40 | ? |
| Angry recall | 0.67 | ? |

**Decision:**
- If test acc < 0.55 → keep YAMNet (CNN+BiLSTM underperformed)
- If 0.55 ≤ test acc < 0.65 → compare live mic; whichever *feels* better wins
- If test acc ≥ 0.65 → switch to CNN+BiLSTM